# 04. S0/S1/S2 Normalization

이 노트북은 Track B의 세 가지 LLM normalizer 조건을 같은 audit case에 대해 실행합니다.

조건:

- S0 `pure`: target token만 보고 정규화
- S1 `pair_fewshot`: raw→norm retrieval pair를 few-shot 예시로 사용
- S2 `metadata_rag`: retrieval pair와 metadata card를 함께 사용

공통 규칙은 target token 하나만 정규화하고, 문장 전체를 다시 쓰지 않는 것입니다. 기본값은 `LIMIT = 20` smoke run입니다.

출력:

- `outputs/context_rag/preds_pure.csv`
- `outputs/context_rag/preds_pair.csv`
- `outputs/context_rag/preds_metadata.csv`


## normalizer 실행 설정

OpenAI model과 smoke run limit을 설정합니다. 이 노트북은 `run_normalizer()`를 직접 호출하며, 같은 작업은 `scripts/04_run_normalizer.py`로 command 실행할 수 있습니다.


In [ ]:
import pathlib, runpy

BOOTSTRAP = pathlib.Path("scripts/01_02_03_04_05_06_notebook_bootstrap.py")
if not BOOTSTRAP.exists():
    BOOTSTRAP = pathlib.Path("/content/lexnorm_submit/scripts/01_02_03_04_05_06_notebook_bootstrap.py")

setup_project = runpy.run_path(str(BOOTSTRAP))["setup_project"]
PROJECT_ROOT = setup_project()

from lexnorm.utils import sync_to_drive

from lexnorm.rag import run_normalizer

MODEL = "gpt-4.1-mini"
LIMIT = 20
CASES_CSV = "outputs/context_rag/audit_cases.csv"
INDEX_CSV = "outputs/context_rag/retrieval_index.csv"
METADATA_JSONL = "outputs/context_rag/metadata_cards.jsonl"
SCHEMA = pathlib.Path("schemas/04_normalizer_schema.json")
PROMPTS = {
    "pure": pathlib.Path("prompts/04_normalizer_pure.txt"),
    "pair_fewshot": pathlib.Path("prompts/04_normalizer_pair_fewshot.txt"),
    "metadata_rag": pathlib.Path("prompts/04_normalizer_metadata_rag.txt"),
}
OUTPUTS = {
    "pure": "outputs/context_rag/preds_pure.csv",
    "pair_fewshot": "outputs/context_rag/preds_pair.csv",
    "metadata_rag": "outputs/context_rag/preds_metadata.csv",
}


## S0/S1/S2 normalizer 실행

세 조건을 같은 audit case에 대해 순서대로 실행하고 prediction CSV를 저장합니다. OpenAI API 호출이므로 `OPENAI_API_KEY`가 필요합니다.


In [ ]:
prediction_dfs = {}
for mode, output_csv in OUTPUTS.items():
    print("=" * 80)
    print("running", mode, "->", output_csv)
    df = run_normalizer(
        cases_csv=CASES_CSV,
        output_csv=output_csv,
        prompt_path=PROMPTS[mode],
        schema_path=SCHEMA,
        mode=mode,
        model=MODEL,
        index_csv=INDEX_CSV,
        metadata_jsonl=METADATA_JSONL,
        softening_policy="preserve_force",
        limit=LIMIT,
    )
    prediction_dfs[mode] = df
    print("saved", output_csv, len(df))
    display(df.head())
    sync_to_drive(output_csv)
